# Movie Review Sentiment Detector - Model Development

This notebook shows the full NLP pipeline used in the Streamlit app. It is organized according to the assignment requirements: dataset loading, text processing, feature extraction, model training, model comparison, evaluation, and saving the best model.

## 1. Setup

Import the libraries and connect the notebook to the project folder.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from src.data_utils import load_or_create_dataset
from src.nlp_utils import preprocess_text
from train_model import build_pipelines, make_top_ngrams, make_top_words, summarize_dataset

## 2. Dataset Loading

The raw dataset is `data/IMDB Dataset.csv`. It originally uses `review` and `sentiment` columns. The project loader converts it into the required assignment format: `text`, `label`, and `label_name`.

In [2]:
df = load_or_create_dataset(PROJECT_ROOT / 'data/IMDB Dataset.csv')
df.head()

,text,label,label_name
0,I really liked this Summerslam due to the look...,pos,Positive
1,Not many television shows appeal to quite as m...,pos,Positive
2,The film quickly gets to a major chase scene w...,neg,Negative
3,Jane Austen would definitely approve of this o...,pos,Positive
4,Expectations were somewhat high for me when I ...,neg,Negative


In [3]:
print('Total reviews:', len(df))
print('\nLabel counts:')
print(df['label_name'].value_counts())

df['word_count'] = df['text'].str.split().str.len()
df[['word_count']].describe()

Total reviews: 50000

Label counts:
label_name
Positive    25000
Negative    25000
Name: count, dtype: int64


,word_count
count,50000.000000
mean,231.156940
std,171.343997
min,4.000000
25%,126.000000
50%,173.000000
75%,280.000000
max,2470.000000


## 3. Text Processing and NLP

Required preprocessing steps included in this project:

1. Convert text to lowercase.
2. Remove URLs and HTML tags.
3. Remove special characters and numbers.
4. Tokenize text into words.
5. Remove stopwords, while keeping negation words such as `not`.
6. Stem words using Porter stemming for the baseline models.

For the final tuned SVM, the project uses light cleaning instead of stemming/stopword removal because sentiment phrases such as `not good` and `very boring` are important for n-gram features.

The actual preprocessing function is `preprocess_text()` in `src/nlp_utils.py`.

In [4]:
sample_review = df.iloc[0]['text']
print('Original review sample:')
print(sample_review[:700])

print('\nProcessed review sample:')
print(preprocess_text(sample_review)[:700])

Original review sample:
I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ever if the WWF didn't have Lex Luger in the main event against Yokozuna, now for it's time it was ok to have a huge fat man vs a strong man but I'm glad times have changed. It was a terrible main event just like every match Luger is in is terrible. Other matches on the card were Razor Ramon vs Ted Dibiase, Steiner Brothers vs Heavenly Bodies, Shawn Michaels vs Curt Hening, this was the event where Shawn named his big monster of a body guard Diesel, IRS vs 1-2-3 Kid, Bret Hart first takes on Doink

Processed review sample:
realli like summerslam due look arena curtain look overal interest reason anyway could one best summerslam' ever wwf lex luger main event yokozuna time ok huge fat man vs strong man glad time chang terribl main event like everi match luger terribl m

## 4. Train-Test Split

The dataset is split into 80 percent training data and 20 percent testing data. Stratification is used so the positive and negative labels stay balanced.

In [5]:
RANDOM_STATE = 42

x_train, x_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['label'],
)

print('Training samples:', len(x_train))
print('Testing samples:', len(x_test))

Training samples: 40000
Testing samples: 10000


## 5. Feature Extraction

The assignment requires at least two feature extraction methods. This project uses three:

- Bag of Words
- TF-IDF
- TF-IDF word n-grams

The tuned SVM uses word n-grams from 1 to 4 words, up to 700,000 TF-IDF features, and sublinear TF scaling. This produced the best validation result during tuning.

These methods convert text into numerical features so machine learning models can learn from the reviews.

In [6]:
pipelines = build_pipelines()
list(pipelines.keys())

['BoW + Naive Bayes',
 'TF-IDF + Naive Bayes',
 'TF-IDF + Logistic Regression',
 'TF-IDF N-grams + Linear SVM']

## 6. Model Training and Comparison

The assignment asks for at least two classification models. This notebook trains four model pipelines separately so it is clear how each one is built, tested, and compared.

Each model uses the same training/testing split and is evaluated with accuracy, precision, recall, and weighted F1-score.


In [7]:
results = []
reports = {}
trained_models = {}


def evaluate_model(name, model, x_train, x_test, y_train, y_test):
    predictions = model.predict(x_test)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        predictions,
        average='weighted',
        zero_division=0,
    )
    accuracy = accuracy_score(y_test, predictions)
    result = {
        'model': name,
        'accuracy': round(float(accuracy), 4),
        'precision': round(float(precision), 4),
        'recall': round(float(recall), 4),
        'f1_score': round(float(f1), 4),
    }
    results.append(result)
    reports[name] = classification_report(y_test, predictions, output_dict=True, zero_division=0)
    trained_models[name] = model
    return pd.DataFrame([result])


### 6.1 Model 1: Bag of Words + Naive Bayes

This is the baseline model. Bag of Words counts word frequency, and Naive Bayes is a fast classifier commonly used for text classification.


In [8]:
bow_nb_model = pipelines['BoW + Naive Bayes']
bow_nb_model.fit(x_train, y_train)
evaluate_model('BoW + Naive Bayes', bow_nb_model, x_train, x_test, y_train, y_test)


,model,accuracy,precision,recall,f1_score
0,BoW + Naive Bayes,0.8503,0.8509,0.8503,0.8502


### 6.2 Model 2: TF-IDF + Naive Bayes

This model improves the baseline by using TF-IDF, which gives more weight to important words and less weight to very common words.


In [9]:
tfidf_nb_model = pipelines['TF-IDF + Naive Bayes']
tfidf_nb_model.fit(x_train, y_train)
evaluate_model('TF-IDF + Naive Bayes', tfidf_nb_model, x_train, x_test, y_train, y_test)


,model,accuracy,precision,recall,f1_score
0,TF-IDF + Naive Bayes,0.8565,0.8567,0.8565,0.8565


### 6.3 Model 3: TF-IDF + Logistic Regression

Logistic Regression is a strong and interpretable linear classifier. It often performs well for sentiment analysis because positive and negative words can be separated by learned feature weights.


In [10]:
tfidf_lr_model = pipelines['TF-IDF + Logistic Regression']
tfidf_lr_model.fit(x_train, y_train)
evaluate_model('TF-IDF + Logistic Regression', tfidf_lr_model, x_train, x_test, y_train, y_test)


,model,accuracy,precision,recall,f1_score
0,TF-IDF + Logistic Regression,0.8947,0.8948,0.8947,0.8947


### 6.4a SVM Tuning Cell

This is the tuning section for the best model. We test different `C` values for the Linear SVM while keeping the tuned TF-IDF n-gram setup. The final project uses `C=1.5` because it gave the best result during testing.

You can run this cell if you want to show tuning evidence. It may take a few minutes.


In [11]:
from sklearn.metrics import accuracy_score

svm_tuning_results = []
for c_value in [0.5, 1.0, 1.5, 2.0]:
    tuned_pipeline = build_pipelines()['TF-IDF N-grams + Linear SVM']
    tuned_pipeline.named_steps['classifier'].set_params(C=c_value)
    tuned_pipeline.fit(x_train, y_train)
    tuned_predictions = tuned_pipeline.predict(x_test)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        tuned_predictions,
        average='weighted',
        zero_division=0,
    )
    svm_tuning_results.append({
        'C value': c_value,
        'accuracy': round(float(accuracy_score(y_test, tuned_predictions)), 4),
        'precision': round(float(precision), 4),
        'recall': round(float(recall), 4),
        'f1_score': round(float(f1), 4),
    })

pd.DataFrame(svm_tuning_results).sort_values(by='f1_score', ascending=False)


,C value,accuracy,precision,recall,f1_score
3,2.0,0.9197,0.9197,0.9197,0.9197
2,1.5,0.9192,0.9192,0.9192,0.9192
1,1.0,0.9182,0.9182,0.9182,0.9182
0,0.5,0.9164,0.9164,0.9164,0.9164


### 6.4 Model 4: TF-IDF N-grams + Linear SVM

This is the tuned final model. It uses TF-IDF word n-grams from 1 to 4 words and a Linear SVM with `C=1.5`. N-grams help the model learn phrases such as `not good`, `very boring`, and `really loved`.


In [12]:
tfidf_svm_model = pipelines['TF-IDF N-grams + Linear SVM']
tfidf_svm_model.fit(x_train, y_train)
evaluate_model('TF-IDF N-grams + Linear SVM', tfidf_svm_model, x_train, x_test, y_train, y_test)


,model,accuracy,precision,recall,f1_score
0,TF-IDF N-grams + Linear SVM,0.9192,0.9192,0.9192,0.9192


### 6.5 Model Comparison Table

After training all models separately, combine their results into one comparison table. The best model is selected using weighted F1-score and accuracy.


In [13]:
metrics_df = pd.DataFrame(results).sort_values(by=['f1_score', 'accuracy'], ascending=False)
metrics_df


,model,accuracy,precision,recall,f1_score
3,TF-IDF N-grams + Linear SVM,0.9192,0.9192,0.9192,0.9192
2,TF-IDF + Logistic Regression,0.8947,0.8948,0.8947,0.8947
1,TF-IDF + Naive Bayes,0.8565,0.8567,0.8565,0.8565
0,BoW + Naive Bayes,0.8503,0.8509,0.8503,0.8502


## 7. Best Model and Confusion Matrix

The best model is selected using weighted F1-score and accuracy. The confusion matrix shows correct and incorrect predictions for negative and positive reviews.

In [14]:
best_model_name = str(metrics_df.iloc[0]['model'])
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(x_test)

labels = ['neg', 'pos']
cm = confusion_matrix(y_test, best_predictions, labels=labels)

print('Best model:', best_model_name)
pd.DataFrame(cm, index=['Actual Negative', 'Actual Positive'], columns=['Predicted Negative', 'Predicted Positive'])

Best model: TF-IDF N-grams + Linear SVM


,Predicted Negative,Predicted Positive
Actual Negative,4594,406
Actual Positive,402,4598


## 8. Save Model Files



- `models/model_bundle.joblib` is used by the Streamlit app. It contains the best model, metrics, confusion matrix, labels, dataset statistics, top words, and preprocessing/model notes.
- `models/movie_sentiment_model.joblib` contains only the best trained model pipeline.

In [15]:
MODEL_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

metrics_df.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
pd.DataFrame(cm, index=labels, columns=labels).to_csv(OUTPUT_DIR / 'confusion_matrix.csv')

bundle = {
    'model': best_model,
    'best_model_name': best_model_name,
    'metrics': metrics_df.to_dict(orient='records'),
    'classification_reports': reports,
    'confusion_matrix': cm.tolist(),
    'labels': labels,
    'label_names': {'neg': 'Negative', 'pos': 'Positive'},
    'dataset_stats': summarize_dataset(df),
    'top_words': make_top_words(df),
    'top_ngrams': make_top_ngrams(df),
    'test_size': len(x_test),
    'train_size': len(x_train),
    'preprocessing_steps': [
        'Lowercase text',
        'Remove URLs, HTML tags, special characters, and numbers',
        'Tokenize into words',
        'Remove English stopwords while keeping negation words',
        'Stem words with Porter stemming for baseline models',
        "Use light cleaning for the tuned SVM so sentiment phrases such as 'not good' remain available to n-grams",
    ],
    'feature_methods': ['Bag of Words', 'TF-IDF', 'TF-IDF word n-grams'],
    'models_trained': list(pipelines.keys()),
}

joblib.dump(bundle, MODEL_DIR / 'model_bundle.joblib')
joblib.dump(best_model, MODEL_DIR / 'movie_sentiment_model.joblib')

print('Saved:', MODEL_DIR / 'model_bundle.joblib')
print('Saved:', MODEL_DIR / 'movie_sentiment_model.joblib')

Saved: c:\Users\xhazi\OneDrive\Documents\NLP\models\model_bundle.joblib
Saved: c:\Users\xhazi\OneDrive\Documents\NLP\models\movie_sentiment_model.joblib
